# 10_2. Theta bound offline 분석 — SA trajectory

theta encoding 구간을 바꿨을 때 **QUBO 구조가 어떻게 달라지는지** 를 기존 결과만으로
계산한다. SA, QPU, 실제 minor embedding 은 실행하지 않는다.

163~240 iteration. Gurobi 와 QA 사이의 중간 구간을 메운다.

## trajectory 를 나누는 이유

cut 은 trajectory 마다 다르다. Gurobi 의 25번째와 QA 의 25번째는 **서로 다른 cut
집합**이므로 직접 비교하면 안 된다. 종합 비교는 `10_5` 에서 한다.

## Bound 전략

| ID | 이름 | θ_L | θ_U |
|----|------|-----|-----|
| S0 | Baseline | 0 | Σ f_j + Σ Σ c_ij d_i |
| S1 | Cost-Based | 0 | Σ d_i max_j c_ij |
| S2 | Assignment Relaxation | Σ d_i min_j c_ij | Σ d_i max_j c_ij |
| S3 | All-Open LP | Q(1) | Σ d_i max_j c_ij |

## 실험

| | 고정 | 묻는 것 |
|---|---|---|
| **E1-A** | b = 10 | 같은 bit 로 얼마나 더 정밀해지는가 |
| **E1-B** | δ_ref = S0 의 δ_θ | 같은 정밀도에 bit 가 몇 개 필요한가 |
| **E2** | — (b 4~12) | 정밀도와 복잡도의 trade-off |

각 실험을 **전 구간(iteration 별)** 과 **최종 상태** 둘 다 계산하고,
실제 QUBO 는 대표 snapshot 에서만 만든다.

**E1-B 주의.** `ceil` 때문에 실제 δ_θ 는 목표값과 같지 않고 **이하**이다.
`δ_cut = 0.25 × 실제 δ_θ` 규칙은 그대로다.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from cflp_bd_qa_ex2.config.loader import load_config
from cflp_bd_qa_ex2.data.io import load_instance
from cflp_bd_qa_ex2.evaluation.bound_analysis import analysis_config_hash

cfg = load_config(ROOT / "config" / "experiments" / "pilot.yaml")
RESULTS = ROOT / "results"
OUT_DIR = RESULTS / "theta_bound"

TRAJECTORY_SOURCE = "sa"

# 25x50 벤치마크 5개만 대상으로 한다.
INSTANCE_IDS = ["25x50_b81", "25x50_b82", "25x50_b83", "25x50_b84", "25x50_b91"]
instances = {tid: load_instance(ROOT / "data" / "benchmark" / f"{tid}.json")
              for tid in INSTANCE_IDS}

# ===== 분석 설정 =====
# configuration_hash 는 실험 설정만 반영하므로, 아래 값을 바꿔도 변하지 않는다.
# 서로 다른 분석이 같은 merge key 로 덮어쓰지 않도록 별도 해시를 만든다.
THETA_BITS_FIXED = 10
BIT_SWEEP = [4, 6, 8, 10, 12]
CUT_SLACK_RATIO = float(cfg["encoding"]["cut_slack"]["delta_ratio_to_theta"])

# 대표 snapshot 에서만 실제 QUBO 를 만든다. 비용이 크기 때문이다.
SNAPSHOT_POLICY = "last_executed_and_early"
SNAPSHOT_ITERATIONS = [1, 5, 10, 25]   # 여기에 "마지막 실행 MP" 가 추가된다

ANALYSIS_HASH = analysis_config_hash(
    cut_slack_ratio=CUT_SLACK_RATIO,
    reference_bits=THETA_BITS_FIXED,
    bit_sweep=BIT_SWEEP,
    snapshot_policy=SNAPSHOT_POLICY,
    snapshot_iterations=SNAPSHOT_ITERATIONS,
)

print("trajectory source :", TRAJECTORY_SOURCE)
print("instances         :", INSTANCE_IDS)
print("cut slack ratio   :", CUT_SLACK_RATIO)
print("analysis hash     :", ANALYSIS_HASH)

# 이전 분석 결과와 섞이지 않는 버전별 출력 경로
OUT_DIR = OUT_DIR / "analysis_v2" / ANALYSIS_HASH
SOURCE_CONFIGURATION_HASH = None
SOURCE_GRAPH_ID = None
NEGATIVE_TOLERANCE = float(cfg["encoding"]["cut_slack"].get("negative_tolerance", 1e-6))


## §1. Bound 계산 및 검증

S3 의 `Q(1)` 은 모든 시설을 연 transportation **LP** 다. 25×50(변수 1,250개)이면
제한 라이선스로도 풀린다.

검증은 기준 해에서 `θ_L ≤ Q(y) ≤ θ_U` 를 확인하는 것이다. **증명이 아니라 구현
검증이다.** ground truth 가 `OPTIMAL` 이 아니면 incumbent 로 검사하고 그 사실을
`reference_kind` 에 남긴다.

In [ ]:
from cflp_bd_qa_ex2.encoding.bounds import STRATEGY_IDS, build_all_bounds, validate_bound
from cflp_bd_qa_ex2.models.cflp import solve_original_cflp
from cflp_bd_qa_ex2.models.subproblem import solve_subproblem
from cflp_bd_qa_ex2.status import Status

bounds = {tid: build_all_bounds(inst, cfg) for tid, inst in instances.items()}
bound_rows = [bounds[tid][sid].as_record() for tid in INSTANCE_IDS for sid in STRATEGY_IDS]
display(pd.DataFrame(bound_rows)[
    ["instance_id", "bound_strategy", "theta_lower_bound", "theta_upper_bound",
     "theta_range", "bound_computation_time"]])


In [ ]:
validation_rows = []
for tid, inst in instances.items():
    ground_truth = solve_original_cflp(inst, cfg)
    reference_kind = "optimal" if ground_truth.status is Status.OPTIMAL else "incumbent"

    y_reference = ground_truth.payload.get("y")
    if y_reference is None:
        print(f"{tid}: 기준 해가 없다 ({ground_truth.status}). 검증을 건너뛴다")
        recourse = float("nan")
    else:
        sp_outcome, _ = solve_subproblem(inst, np.asarray(y_reference, dtype=int), cfg)
        if sp_outcome.status is not Status.OPTIMAL or sp_outcome.objective is None:
            print(f"{tid}: SP 가 OPTIMAL 이 아니다 ({sp_outcome.status}). 검증을 건너뛴다")
            recourse = float("nan")
        else:
            recourse = float(sp_outcome.objective)

    for sid in STRATEGY_IDS:
        row = validate_bound(bounds[tid][sid], recourse, reference_kind=reference_kind)
        row["ground_truth_status"] = str(ground_truth.status)
        validation_rows.append(row)

validation = pd.DataFrame(validation_rows)
checked = validation[validation["validation_status"] == "checked"]
not_checked = validation[validation["validation_status"] != "checked"]
clipping = int(checked["theta_clipping_count"].sum()) if len(checked) else 0

print(f"검사한 조합 {len(checked)} / 건너뛴 조합 {len(not_checked)}")
print(f"theta_clipping_count 합계: {clipping} (0 이어야 한다)")
if clipping:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    validation.to_csv(OUT_DIR / f"invalid_bounds_{TRAJECTORY_SOURCE}.csv", index=False)
    raise ValueError("bound 위반을 발견하여 분석을 중단한다. 진단 CSV를 확인하라")
display(validation[["instance_id", "bound_strategy", "reference_kind",
                    "recourse_at_reference", "lower_slack", "upper_slack",
                    "bound_valid", "validation_status"]])


## §2. trajectory 복원과 분석 시점

`activated=True` 인 cut 을 전부 쓰면 안 된다. 종료 iteration 에서 생성된 cut 은
**다음 iteration 이 실행되지 않아 MP 에 들어간 적이 없다.**
`MAX_ITERATIONS` 로 끝난 Gurobi 에서 특히 중요하다.

`active_cuts_at(cuts, last_iteration)` 로 **마지막으로 실행된 MP** 를 쓴다.
종료 직후의 전체 cut pool 은 별도 snapshot 으로 표시한다.

In [ ]:
from cflp_bd_qa_ex2.evaluation.bound_analysis import (
    active_cuts_at, analyze_configuration, analyze_trajectory,
    load_observed_trajectory, verify_snapshot_qubo,
)

trajectories, executed, snapshot_info, observed = {}, {}, [], {}
for tid in INSTANCE_IDS:
    trajectory = load_observed_trajectory(
        RESULTS, TRAJECTORY_SOURCE, tid,
        configuration_hash=SOURCE_CONFIGURATION_HASH, graph_id=SOURCE_GRAPH_ID,
    )
    observed[tid] = trajectory
    trajectories[tid] = list(trajectory.cuts)
    executed[tid] = active_cuts_at(trajectory.cuts, trajectory.last_iteration)
    snapshot_info.append({
        "instance_id": tid,
        "cuts_recorded": len(trajectory.cuts),
        "cuts_activated": sum(c.activated for c in trajectory.cuts),
        "cuts_in_last_executed_mp": len(executed[tid]),
        "last_iteration": trajectory.last_iteration,
        "trajectory_hash": trajectory.trajectory_hash,
    })

display(pd.DataFrame(snapshot_info))
print("분석 시점은 원본 iteration CSV에 기록된 실행 MP 시작 시점이다")


## §3. E1-A — Fixed Bit Budget (b = 10)

θ bit 를 고정하므로 **θ 변수 수는 변하지 않는다.** 바뀌는 것은 δ_θ, δ_cut,
cut slack bit 다.

In [ ]:
def rows_for(tid, bound, cuts, snapshot, **kwargs):
    """한 설정의 최종 상태 행을 만든다."""
    row = analyze_configuration(instances[tid], bound, cuts,
                                cut_slack_ratio=CUT_SLACK_RATIO,
                                negative_tolerance=NEGATIVE_TOLERANCE, **kwargs)
    row.update({
        "trajectory_source": TRAJECTORY_SOURCE,
        "snapshot": snapshot,
        "analysis_hash": ANALYSIS_HASH,
        "benders_iteration": observed[tid].last_iteration,
        "trajectory_hash": observed[tid].trajectory_hash,
    })
    return row

e1a_rows = [
    rows_for(tid, bounds[tid][sid], executed[tid], "last_executed",
             theta_bits=THETA_BITS_FIXED)
    for tid in INSTANCE_IDS for sid in STRATEGY_IDS
]
e1a = pd.DataFrame(e1a_rows)
e1a["experiment"] = "E1-A"
display(e1a[["instance_id", "bound_strategy", "num_cuts", "delta_theta", "delta_cut",
             "cut_slack_bits_total", "logical_variables", "logical_edges",
             "num_cuts_with_negative_a"]])


## §4. E1-B — Fixed Target Resolution

목표 해상도를 **S0 의 δ_θ** 로 두고 필요한 최소 bit 수를 구한다.
실제 δ_θ 는 목표값 **이하**이며 같지 않다.

In [ ]:
e1b_rows = []
for tid in INSTANCE_IDS:
    delta_ref = analyze_configuration(
        instances[tid], bounds[tid]["S0"], executed[tid],
        theta_bits=THETA_BITS_FIXED, cut_slack_ratio=CUT_SLACK_RATIO,
        compute_edges=False,
    )["delta_theta"]

    for sid in STRATEGY_IDS:
        row = rows_for(tid, bounds[tid][sid], executed[tid], "last_executed",
                       target_delta=delta_ref)
        row["delta_ref"] = delta_ref
        row["resolution_margin"] = delta_ref - row["delta_theta"]
        e1b_rows.append(row)

e1b = pd.DataFrame(e1b_rows)
e1b["experiment"] = "E1-B"
display(e1b[["instance_id", "bound_strategy", "theta_bits", "delta_ref", "delta_theta",
             "resolution_margin", "cut_slack_bits_total", "logical_variables"]])


## §5. E2 — Bit Precision Sweep

**b 를 줄이면 QUBO 가 두 방향에서 작아진다** — θ 자체가 줄고, δ_θ 가 커져
δ_cut 도 커지므로 cut slack bit 도 줄어든다. 해상도가 거칠어지는 대가이며,
discretization 손실은 여기서 측정하지 않는다.

In [ ]:
e2_rows = [
    rows_for(tid, bounds[tid][sid], executed[tid], "last_executed", theta_bits=bits)
    for tid in INSTANCE_IDS for sid in STRATEGY_IDS for bits in BIT_SWEEP
]
e2 = pd.DataFrame(e2_rows)
e2["experiment"] = "E2"
display(e2.pivot_table(index=["instance_id", "bound_strategy"],
                       columns="theta_bits", values="logical_variables"))


## §6. 전 구간 분석 (iteration 별 자원 증가)

cut 하나의 slack bit 수는 iteration 과 무관하므로 **한 번 계산해 누적합**으로
모든 iteration 을 얻는다. edge 는 여기서 계산하지 않는다 — 대표 snapshot 에서만
실제 QUBO 로 구한다.

In [ ]:
trajectory_rows = []
for tid in INSTANCE_IDS:
    delta_ref = analyze_configuration(
        instances[tid], bounds[tid]["S0"], executed[tid],
        theta_bits=THETA_BITS_FIXED, cut_slack_ratio=CUT_SLACK_RATIO,
        compute_edges=False,
    )["delta_theta"]

    for sid in STRATEGY_IDS:
        for experiment, kwargs in (
            ("E1-A", {"theta_bits": THETA_BITS_FIXED}),
            ("E1-B", {"target_delta": delta_ref}),
        ):
            for row in analyze_trajectory(
                instances[tid], bounds[tid][sid], trajectories[tid],
                cut_slack_ratio=CUT_SLACK_RATIO,
                observations=observed[tid].observations,
                negative_tolerance=NEGATIVE_TOLERANCE, **kwargs
            ):
                row.update({
                    "trajectory_source": TRAJECTORY_SOURCE,
                    "experiment": experiment,
                    "snapshot": "per_iteration",
                    "analysis_hash": ANALYSIS_HASH,
                })
                trajectory_rows.append(row)

per_iteration = pd.DataFrame(trajectory_rows)
print(f"전 구간 {len(per_iteration)}행")

sample = per_iteration[(per_iteration.instance_id == INSTANCE_IDS[0])
                       & (per_iteration.experiment == "E1-A")]
display(sample.pivot_table(index="benders_iteration", columns="bound_strategy",
                           values="logical_variables").tail(10))


## §7. 대표 snapshot — 실제 QUBO 검증

수식 분석은 변수와 edge 를 예측할 뿐 **실제 계수 범위는 모른다.**
사전에 정한 iteration 에서 실제 QUBO 를 만들어 대조하고 coefficient range 를 잰다.

In [ ]:
snapshot_rows = []
for tid in INSTANCE_IDS:
    last_iteration = observed[tid].last_iteration
    targets = [i for i in SNAPSHOT_ITERATIONS if i <= last_iteration]
    if last_iteration and last_iteration not in targets:
        targets.append(last_iteration)

    for iteration in targets:
        cuts = active_cuts_at(trajectories[tid], iteration)
        for sid in STRATEGY_IDS:
            row = verify_snapshot_qubo(
                instances[tid], bounds[tid][sid], cuts, cfg,
                theta_bits=THETA_BITS_FIXED, cut_slack_ratio=CUT_SLACK_RATIO,
            )
            row.update({
                "trajectory_source": TRAJECTORY_SOURCE,
                "benders_iteration": iteration,
                "snapshot": "representative",
                "experiment": "E1-A",
                "analysis_hash": ANALYSIS_HASH,
            })
            snapshot_rows.append(row)

snapshots = pd.DataFrame(snapshot_rows)
mismatch = snapshots[~(snapshots.variables_match & snapshots.edges_match)]
print(f"수식 분석과 실제 QUBO 불일치: {len(mismatch)}건 (0 이어야 한다)")
if len(mismatch):
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    mismatch.to_csv(OUT_DIR / f"invalid_snapshots_{TRAJECTORY_SOURCE}.csv", index=False)
    raise ValueError("수식과 실제 QUBO가 달라 분석을 중단한다. 진단 CSV를 확인하라")
display(snapshots[["instance_id", "bound_strategy", "benders_iteration", "num_cuts",
                   "actual_variables", "actual_edges", "qubo_density",
                   "qubo_log10_coefficient_range"]])


## §8. Bound–Slack Encoding Relationship

### 검증할 정리

δ_cut = r (θ_U − θ_L)/(2^b − 1) 이고 S_k^max = θ_U − A_k 이므로, b 와 cut pool 을
고정하면

```
θ_L 증가  ⇒  δ_cut 감소  ⇒  b_k 증가 또는 유지
```

따라서 **S1 → S2 → S3 에서 cut slack bit 총합은 감소할 수 없다.**
반면 **S0 → S1 은 θ_U 가 바뀌므로 증가·감소·유지 모두 가능하다.**

`A_k < 0` 인 cut 에서는 θ_U 를 줄일 때 **S_k^max 자체는 줄어든다.**
늘어나는 것은 필요한 격자 수 `S_k^max / δ_cut` 이고, 그 결과 bit 가 증가할 수 있다.

In [ ]:
checks = []
for tid in INSTANCE_IDS:
    totals = e1a[e1a.instance_id == tid].set_index("bound_strategy")["cut_slack_bits_total"]
    checks.append({
        "instance_id": tid,
        **{sid: int(totals[sid]) for sid in STRATEGY_IDS},
        "S1_to_S3_non_decreasing": bool(totals["S1"] <= totals["S2"] <= totals["S3"]),
        "S0_to_S1_change": int(totals["S1"] - totals["S0"]),
    })

monotonicity = pd.DataFrame(checks)
print("정리 위반:", int((~monotonicity["S1_to_S3_non_decreasing"]).sum()), "건 (0 이어야 한다)")
display(monotonicity)

negative = e1a[e1a.bound_strategy == "S1"][
    ["instance_id", "num_cuts", "num_cuts_with_negative_a", "a_min", "a_max"]].copy()
negative["negative_ratio"] = negative["num_cuts_with_negative_a"] / negative["num_cuts"]
display(negative)

if not monotonicity["S1_to_S3_non_decreasing"].all():
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    monotonicity.to_csv(OUT_DIR / f"invalid_monotonicity_{TRAJECTORY_SOURCE}.csv", index=False)
    raise ValueError("단조성 검증에 실패하여 분석을 중단한다")


## §9. Offline Resource Trade-off Analysis

**자원과 정밀도만 본다.** raw feasibility, 실제 QA solution gap, embedding time 은
별도 실험이 필요하며 여기서 주장할 수 없다.

정밀도 축은 **절대 δ_θ** 를 쓴다. `delta_theta / theta_range` 는 b 에만 의존해
(= 1/(2^b − 1)) bound 전략 간 차이가 사라지므로 쓰지 않는다.

In [ ]:
final = pd.concat([e1a, e1b, e2], ignore_index=True)

summary = final.groupby(["experiment", "bound_strategy"]).agg(
    theta_bits_mean=("theta_bits", "mean"),
    delta_theta_mean=("delta_theta", "mean"),
    cut_slack_bits_mean=("cut_slack_bits_total", "mean"),
    logical_variables_mean=("logical_variables", "mean"),
    logical_edges_mean=("logical_edges", "mean"),
).reset_index()
display(summary)

baseline = e1a[e1a.bound_strategy == "S0"].set_index("instance_id")["logical_variables"]
for frame, name in ((e1a, "E1-A"), (e1b, "E1-B")):
    work = frame.copy()
    work["baseline_variables"] = work["instance_id"].map(baseline)
    work["change_pct"] = 100 * (work["logical_variables"] - work["baseline_variables"]) \
                         / work["baseline_variables"]
    print(f"\n{name}: S0(b=10) 대비 logical variable 변화율 (%)")
    display(work.groupby("bound_strategy")["change_pct"].mean().round(2))


In [ ]:
from cflp_bd_qa_ex2.evaluation.results import attach_provenance, save_records

# 모든 결과 종류에 동일한 분석 설정과 입력 trajectory 식별자를 보존한다.
for frame in (final, per_iteration, snapshots):
    frame["analysis_hash"] = ANALYSIS_HASH
    frame["trajectory_hash"] = frame["instance_id"].map(
        {tid: trajectory.trajectory_hash for tid, trajectory in observed.items()})
for rows in (bound_rows, validation_rows):
    for row in rows:
        row["analysis_hash"] = ANALYSIS_HASH
        row["trajectory_hash"] = observed[row["instance_id"]].trajectory_hash

formats = cfg["output"]["formats"]
# 분석 설정과 snapshot 을 merge key 에 넣는다.
# configuration_hash 만으로는 ratio/bit sweep 변경이 구분되지 않아 덮어써진다.
merge_keys = ("instance_id", "trajectory_source", "experiment", "bound_strategy",
              "theta_bits", "snapshot", "benders_iteration",
              "configuration_hash", "analysis_hash", "trajectory_hash")

save_records(attach_provenance(final.to_dict("records"), cfg),
             OUT_DIR, f"theta_bound_{TRAJECTORY_SOURCE}", formats, merge_keys=merge_keys)
save_records(attach_provenance(per_iteration.to_dict("records"), cfg),
             OUT_DIR, f"theta_bound_per_iteration_{TRAJECTORY_SOURCE}", formats,
             merge_keys=merge_keys)
save_records(attach_provenance(snapshots.to_dict("records"), cfg),
             OUT_DIR, f"theta_bound_snapshots_{TRAJECTORY_SOURCE}", formats,
             merge_keys=merge_keys)
save_records(attach_provenance(bound_rows, cfg), OUT_DIR,
             f"theta_bound_definitions_{TRAJECTORY_SOURCE}", formats,
             merge_keys=("instance_id", "bound_strategy", "configuration_hash",
                         "analysis_hash", "trajectory_hash"))
save_records(attach_provenance(validation_rows, cfg), OUT_DIR,
             f"theta_bound_validation_{TRAJECTORY_SOURCE}", formats,
             merge_keys=("instance_id", "bound_strategy", "configuration_hash",
                         "analysis_hash", "trajectory_hash"))
print("저장:", OUT_DIR)


## 해석

- **E1-A 에서 S1→S2→S3 의 cut slack bit 가 줄지 않는 것은 버그가 아니라 정리**다.
- **E1-B 에서 θ bit 가 줄어드는 것**은 같은 목표 해상도를 더 좁은 구간으로
  달성하기 때문이다. 실제 δ_θ 는 목표값보다 작으므로 cut slack bit 증감은
  `ceil` 경계에 달려 있고 단조적이지 않다.
- **분석 시점은 "마지막으로 실행된 MP"** 다. 종료 iteration 의 cut 은 MP 에
  들어간 적이 없으므로 기본 분석에서 제외한다.
- logical edges 는 대표 snapshot 에서 **실제 QUBO 와 일치하는지 확인**했다.
  `constraint_interaction_upper_bound` 는 중복을 포함한 상한이며 edge 수가 아니다.
- 이 결과만으로 solution quality 를 주장하지 않는다. `10_2` 의 결론은
  "어떤 조합이 QUBO 구조상 유리한가" 까지다.
